# Import Library

In [1]:
import mlflow
import numpy as np
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load Dataset

In [35]:
df=pd.read_csv("https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/refs/heads/main/tweet_emotions.csv")
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


- drop 'tweet_id'

In [36]:
df.drop(columns='tweet_id',inplace=True)

- Data Preprocessing

In [38]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatization(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_stop_words(text):
    return " ".join([word for word in str(text).split() if word not in stop_words])

def removing_numbers(text):
    return "".join([char for char in text if not char.isdigit()])

def lower_case(text):
    return " ".join([word.lower() for word in text.split()])

def remove_punctuation(text):
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    return text.strip()

def remove_urls(text):
    return re.sub(r"http\S+", "", text).strip()


def normalize_text(df: pd.DataFrame) -> pd.DataFrame:
    try:
        print("Starting text normalization")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(lemmatization)

        print("Text normalization completed")
        return df

    except Exception as e:
        print(f"Error in text normalization: {e}")
        raise

In [39]:
df=normalize_text(df)


Starting text normalization
Text normalization completed


In [6]:
df.sample(5)

,sentiment,content
38751,neutral,morning sorry missing tweet yesterday damohopo...
16204,sadness,lonejohnny awwww ill waiting hope doesnt rain
15946,hate,date script thats really bad midway thru pitch...
39939,sadness,watermelon haha twitter hard though isnt
30185,fun,im thinking im going fun tonightand maybe chan...


In [28]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

- we just focus on two sentiment 'sadness' and 'happiness'

In [40]:
df=df[df['sentiment'].isin(['sadness','happiness'])]

In [41]:
df.sample(5)

,sentiment,content
8565,sadness,myersandchang
16106,sadness,banged elbow bleeding owwiee
29065,happiness,michaelsheen well guess think everything thank...
9131,sadness,im depressed againugh help
19700,sadness,cried


- replace sadness with 0 and happiness with 1

In [42]:
df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)

C:\Users\hp\AppData\Local\Temp\ipykernel_4168\2140401980.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'].replace({'sadness':0,'happiness':1},inplace=True)


- Extract X and y

In [45]:
X=df['content']
y=df['sentiment']

In [26]:
y

40       1
69       1
77       1
126      1
233      1
        ..
39986    1
39987    1
39988    1
39994    1
39998    1
Name: sentiment, Length: 5209, dtype: int64

- Build Pipeline

In [22]:
vectorizer={
    'countvector':CountVectorizer(),
    'tfidfvector':TfidfVectorizer()
}

In [19]:
algorithms={
    'logisticregression':LogisticRegression(),
    'naivebayes':MultinomialNB(),
    'xgboost':XGBClassifier(),
    'randomforest':RandomForestClassifier(),
    'gradientboosting':GradientBoostingClassifier()
}

- connect dagshub

In [16]:
import dagshub
import mlflow
mlflow.set_tracking_uri("https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/")
dagshub.init(repo_owner='rahulpatel16092005', repo_name='mlops-mini-project', mlflow=True)

Accessing as rahulpatel16092005

Initialized MLflow to track repo "rahulpatel16092005/mlops-mini-project"

Repository rahulpatel16092005/mlops-mini-project initialized!

In [17]:
mlflow.set_experiment("BOW vs TFIDF")

2026/03/29 13:10:01 INFO mlflow.tracking.fluent: Experiment with name 'BOW vs TFIDF' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/0e9cabb5cdf846888742fc114e0fb8e4', creation_time=1774770003002, experiment_id='1', last_update_time=1774770003002, lifecycle_stage='active', name='BOW vs TFIDF', tags={}, workspace='default'>

In [46]:
with mlflow.start_run(run_name="All Experiments") as parent_run:
    
    for algo_name,algo in algorithms.items():
        for vectorizer_name,vector in vectorizer.items():
                with mlflow.start_run(run_name=f"{algo_name} with {vectorizer_name}", nested=True) as child_run:
                    x=vector.fit_transform(X)
                    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
                    
                    mlflow.log_param("Vectorizer", vectorizer_name)
                    mlflow.log_param("max_features",1000)
                    mlflow.log_param("test_size",0.2)

                    model = algo
                    model.fit(x_train, y_train)

                    if algo_name=='xgboost':
                         mlflow.log_param("algo", algo_name)
                         mlflow.log_param("n_estimators", model.n_estimators)
                         mlflow.log_param("learning_rate", model.learning_rate) 
                    elif algo_name=='randomforest':
                        mlflow.log_param("algo", algo_name)
                        mlflow.log_param("n_estimators", model.n_estimators)
                        mlflow.log_param("max_depth", model.max_depth)
                    elif algo_name=='gradientboosting':
                        mlflow.log_param("algo", algo_name)
                        mlflow.log_param("n_estimators", model.n_estimators)
                        mlflow.log_param("learning_rate", model.learning_rate)
                    elif algo_name=='logisticregression':   
                        mlflow.log_param("algo", algo_name)
                        mlflow.log_param("C", model.C)
                    elif algo_name=='naivebayes':
                        mlflow.log_param("algo", algo_name)
                        mlflow.log_param("alpha", model.alpha)


                    y_pred = model.predict(x_test)

                    accuracy = accuracy_score(y_test, y_pred)
                    precision = precision_score(y_test, y_pred, average='weighted')
                    recall = recall_score(y_test, y_pred, average='weighted')
                    f1 = f1_score(y_test, y_pred, average='weighted')

                    mlflow.log_metric("accuracy", accuracy)
                    mlflow.log_metric("precision", precision)
                    mlflow.log_metric("recall", recall)
                    mlflow.log_metric("f1_score", f1)

                    mlflow.sklearn.log_model(model, "Logistic Regression baseline model")

                    notebook_path="exp1_baseline_model.ipynb"
                    import os
                    os.system(f"jupyter nbconvert --to script {notebook_path}")
                    mlflow.log_artifact(notebook_path)

2026/03/29 13:29:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 13:29:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logisticregression with countvector at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/d787c0f9ecd54aa89861f5b0a0a0d9e4
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1


2026/03/29 13:29:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 13:29:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run logisticregression with tfidfvector at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/9ee32aef156c4441ad8fbc7602fa0733
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1


2026/03/29 13:30:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 13:30:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run randomforest with tfidfvector at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/aaea46b35f7d41fba8f71113cf3d3de0
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1


2026/03/29 14:39:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 14:39:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gradientboosting with countvector at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/bb7bb9c61bda4991b932e93006f161bd
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1


2026/03/29 14:41:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/29 14:41:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run gradientboosting with tfidfvector at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/4568e3c16e9f4c1590f543e48932493d
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1/runs/f29fe720985347dda61d432432a873d3
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/mlops-mini-project.mlflow/#/experiments/1
